**5. SVM**
---
Propiedad de René Adarme Amado
---
Universidad Antonio Nariño - Tesis de Maestría en Hidrogeología Ambiental
Abril de 2025

# **Librerías**

In [25]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# **Preprocesado**

In [29]:
# 1. Cargar y limpiar la base de datos
base = pd.read_excel("Base_2025_v3.xlsx")
base.drop(columns=['ID', 'Tipo_Estructuras', 'Alturas'], inplace=True)

variables_numericas = base.select_dtypes(include=['float64', 'int64']).columns.tolist()
variables_numericas.remove('Agua')
variables_categoricas = base.select_dtypes(include=['object']).columns.tolist()

X = base.drop(columns=['Agua'])
y = base['Agua'].astype(int)

# 2. Preprocesador
despacho = ColumnTransformer([
    ('numericas', StandardScaler(), variables_numericas), # Escalado de variables numéricas
    ('categoricas', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), variables_categoricas) # Codificación de variables categóricas
])

# 3. Métricas de evaluación
metricas = {
    'Precision': 'precision',
    'Sensibilidad': make_scorer(recall_score),
    'Puntaje F1': 'f1',
    'AUC-ROC': make_scorer(roc_auc_score),
}

# 4. Preparar la validación cruzada estratificada
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)  # 10 folds

#**MODELO 1**

In [30]:
# 5. Modelo 1: SVM con Kernel Lineal
modelo1 = ImbPipeline([
    ('preprocesador', despacho),
    ('smote', SMOTE(random_state=123)),  # Balancear las clases
    ('clasificador', SVC(kernel='linear', class_weight='balanced', random_state=123, probability=True))
])

resultados_modelo1 = cross_validate(modelo1, X, y, cv=skf, scoring=metricas, return_train_score=True)

# **MODELO 2**

In [33]:
# 6. Modelo 2: SVM con Kernel Radial (RBF) y Optimización de Hiperparámetros
param_grid2 = {'clasificador__C': [0.01, 0.1, 1, 10], 'clasificador__gamma': [0.01, 0.1, 1, 10]}
   # parámetro de regularización: C
modelo2 = ImbPipeline([
    ('preprocesador', despacho),
    ('smote', SMOTE(random_state=123)),
    ('clasificador', SVC(kernel='rbf', class_weight='balanced', random_state=123, probability=True))
])

grid_search = GridSearchCV(modelo2, param_grid2, cv=skf, scoring='f1') # Buscar la mejor configuración
grid_search.fit(X, y)

mejores_params_modelo2 = grid_search.best_params_
print("\nMejores hiperparámetros para Modelo 2:", mejores_params_modelo2)

modelo2_optimizado = grid_search.best_estimator_  # El mejor modelo encontrado por GridSearchCV
resultados_modelo2 = cross_validate(modelo2_optimizado, X, y, cv=skf, scoring=metricas, return_train_score=True)


Mejores hiperparámetros para Modelo 2: {'clasificador__C': 0.01, 'clasificador__gamma': 10}


# **MODELO 3**

In [34]:
# Modelo 3: SVM con Kernel Polinomial
param_grid3 = {'clasificador__C': [0.01, 0.1, 1, 10], 'clasificador__degree': [2, 3, 4]} # Agregamos 'degree'
modelo3 = ImbPipeline([
    ('preprocesador', despacho),
    ('smote', SMOTE(random_state=123)),
    ('clasificador', SVC(kernel='poly', class_weight='balanced', random_state=123, probability=True))
])
grid_search_poly = GridSearchCV(modelo3, param_grid3, cv=skf, scoring='f1')
grid_search_poly.fit(X, y)
mejores_params_modelo3 = grid_search_poly.best_params_
print("\nMejores hiperparámetros para Modelo 3 (Polinomial):", mejores_params_modelo3)
modelo3_optimizado = grid_search_poly.best_estimator_
resultados_modelo3 = cross_validate(modelo3_optimizado, X, y, cv=skf, scoring=metricas, return_train_score=True)



Mejores hiperparámetros para Modelo 3 (Polinomial): {'clasificador__C': 10, 'clasificador__degree': 4}


# **MODELO 4**

In [35]:
# Modelo 4: SVM con Kernel Sigmoide
param_grid4 = {'clasificador__C': [0.01, 0.1, 1, 10], 'clasificador__gamma': [0.01, 0.1, 1, 10]}
modelo4 = ImbPipeline([
    ('preprocesador', despacho),
    ('smote', SMOTE(random_state=123)),
    ('clasificador', SVC(kernel='sigmoid', class_weight='balanced', random_state=123, probability=True))
])
grid_search_sigmoid = GridSearchCV(modelo4, param_grid4, cv=skf, scoring='f1')
grid_search_sigmoid.fit(X, y)
mejores_params_modelo4 = grid_search_sigmoid.best_params_
print("\nMejores hiperparámetros para Modelo 4 (Sigmoide):", mejores_params_modelo4)
modelo4_optimizado = grid_search_sigmoid.best_estimator_
resultados_modelo4 = cross_validate(modelo4_optimizado, X, y, cv=skf, scoring=metricas, return_train_score=True)


Mejores hiperparámetros para Modelo 4 (Sigmoide): {'clasificador__C': 1, 'clasificador__gamma': 0.01}


# **Resumen de modelos**

In [36]:
def resumen_modelo(resultados, nombre_modelo):
    print(f"\n--- Resumen Modelo: {nombre_modelo} ---")
    print(f"{'Métrica':<15}{'Entrenamiento(Media)':<20}{'(Desv.Std)':<20}{'Validación(Media)':<20}{'(Desv.Std)'}")
    for metrica, valores in resultados.items():
        if metrica.startswith('train_'):
            metrica_nombre = metrica[6:]
            print(f"{metrica_nombre:<15}{np.mean(valores):<20.4f}{np.std(valores):<20.4f}{resultados['test_'+metrica[6:]].mean():<20.4f}{resultados['test_'+metrica[6:]].std():<20.4f}")

resumen_modelo(resultados_modelo1, "1 - SVM Lineal")
resumen_modelo(resultados_modelo2, "2 - SVM Radial")
resumen_modelo(resultados_modelo3, "3 - SVM Polinomial")
resumen_modelo(resultados_modelo4, "4 - SVM Sigmoide")


--- Resumen Modelo: 1 - SVM Lineal ---
Métrica        Entrenamiento(Media)(Desv.Std)          Validación(Media)   (Desv.Std)
Precision      0.8437              0.0066              0.8192              0.0645              
Sensibilidad   0.5824              0.0254              0.5486              0.0935              
Puntaje F1     0.6888              0.0188              0.6516              0.0758              
AUC-ROC        0.6687              0.0100              0.6333              0.0684              

--- Resumen Modelo: 2 - SVM Radial ---
Métrica        Entrenamiento(Media)(Desv.Std)          Validación(Media)   (Desv.Std)
Precision      0.7447              0.0302              0.6984              0.0188              
Sensibilidad   0.9923              0.0058              0.9813              0.0243              
Puntaje F1     0.8505              0.0178              0.8158              0.0156              
AUC-ROC        0.6072              0.0587              0.5092              0